# Assignment 1: Data Profiling (EDA) Report

**Progress: Tasks A and B assembled; Tasks C and D remain to be added.**

| Member | Matric number | Contribution |
|---|---|---|
| TO COMPLETE | TO COMPLETE | A: Scenario and data source |
| TO COMPLETE | TO COMPLETE | B: Missing values and other quality issues |
| TO COMPLETE | TO COMPLETE | C: Outliers and diagnostic charts |
| TO COMPLETE | TO COMPLETE | D: PII, bias and conclusion |

## 1. Scenario

A multinational fashion retailer wants to compare returns across customer demographic segments and product categories to support inventory planning and reduce return-processing costs. Using synthetic retail data, we assess missing attributes, duplicate transactions, data types, formats, invalid values and potential outliers, with attention to privacy and representation risks. We ask whether the available data is fit for these comparisons and what limitations remain after justified handling.

## 2. Data Source and Run Instructions

The **Global Fashion Retail Sales** dataset by **Ric Gomes** is available on [Kaggle](https://www.kaggle.com/datasets/ricgomes/global-fashion-retail-stores-dataset). The publisher describes it as synthetic data simulating a multinational retailer, rather than observed real-customer transactions. The reported source licence is CC BY 4.0.

We obtain `customers.csv`, `products.csv` and `transactions.csv` using KaggleHub. The download handle below pins version 24, identified in the source metadata during integration; confirm that this matches the team's files. Local files are reused if present, so use an empty `data/` folder for a fresh version-pinned download.

1. Open this notebook in the repository root in GitHub Codespaces or Jupyter.
2. Select a Python 3 kernel with enough memory for the full tables.
3. Run the installation cell, then restart the kernel if requested.
4. Run all cells from top to bottom. Internet access is needed for installation and the first download. Follow KaggleHub's authentication prompt if access requires it.
5. Keep raw CSVs in `data/`, which is excluded from Git. Do not commit credentials.

The customer table has one row per customer; the product table has one row per product; transactions contain sales/return lines. Customer ID and Product ID connect the tables. A transaction's `Size` is not the same field as the product catalogue's `Sizes` list. This report checks the full tables, not a 1,000-row preview.

The original A/B notebooks remain separate working records. Outputs are cleared because this combined notebook has not yet been executed against the full dataset. Run it from top to bottom in Codespaces before saving the verified results.

In [ ]:
%pip install -U kagglehub pandas matplotlib

In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
dataset = "ricgomes/global-fashion-retail-stores-dataset/versions/24"

for filename in ["customers.csv", "products.csv", "transactions.csv"]:
    if not (data_dir / filename).exists():
        kagglehub.dataset_download(dataset, path=filename, output_dir=str(data_dir))
    print(f"Ready: {filename}")

In [ ]:
import pandas as pd

customers = pd.read_csv(
    "data/customers.csv",
    dtype={"Telephone": "string"},
    low_memory=False,
)

products = pd.read_csv(
    "data/products.csv",
    low_memory=False,
)

print(f"Customers: {customers.shape[0]:,} rows, "
      f"{customers.shape[1]} columns")

print(f"Products: {products.shape[0]:,} rows, "
      f"{products.shape[1]} columns")

In [ ]:
# Load the complete transaction table once; preserve the raw CSV on disk.
transactions = pd.read_csv(data_dir / "transactions.csv", low_memory=False)
print(f"Transactions: {len(transactions):,} rows, {transactions.shape[1]} columns")
print("Transaction types:")
display(transactions["Transaction Type"].value_counts(dropna=False))

## 3. Data Profiling and Issue Handling

### 3.1 Missing Values

Count missing values before cleaning. pandas missing-value detection does not by itself identify every possible placeholder or blank string.

In [ ]:
def missing_summary(df):
    summary = pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing (%)": (df.isna().mean() * 100).round(2),
    })

    return summary.sort_values("Missing Count", ascending=False)


customer_missing = missing_summary(customers)
product_missing = missing_summary(products)

print("CUSTOMERS")
display(customer_missing)

print("PRODUCTS")
display(product_missing)

### Customer Missing-Value Findings

Job Title is missing for 584,185 customers (35.55%).
All other customer columns have no missing values recognised
by pandas at this stage.

Proposed handling: retain these customer records and leave
unrecorded occupations as missing. Job Title is not essential
for the planned comparisons by age, gender, and product category.
Dropping these customers would unnecessarily reduce coverage.

If occupation is analysed later, missing occupations should be
reported as a separate "Not recorded" group, rather than guessed.

### Investigating Missing Product Attributes

Color is missing for 12,445 products (69.37%), while Sizes is
missing for 2,070 products (11.54%).

We examine missingness by product subcategory before deciding
how to handle these values. Missing values alone do not establish
whether an attribute is unknown or not applicable.

In [ ]:
product_missing_by_group = (
    products
    .assign(
        color_missing=products["Color"].isna(),
        sizes_missing=products["Sizes"].isna(),
    )
    .groupby(["Category", "Sub Category"])
    .agg(
        product_count=("Product ID", "size"),
        color_missing_pct=("color_missing", "mean"),
        sizes_missing_pct=("sizes_missing", "mean"),
    )
)

percent_columns = ["color_missing_pct", "sizes_missing_pct"]

product_missing_by_group[percent_columns] = (
    product_missing_by_group[percent_columns] * 100
).round(2)

display(product_missing_by_group)

### Product Missing-Value Decisions

All 2,070 missing Sizes values occur in Accessories, with 690
affected products in each main category. No other subcategory
has missing Sizes. This suggests a category-related recording
pattern, but does not confirm that all accessories are one-size.

Color is missing for 69.37% of products, with missingness varying
substantially across subcategories. Dropping these records would
remove some subcategories entirely and distort category coverage.

We retain all products, preserve the missing values, and add
missingness flags. We do not impute colours or sizes because
there is insufficient evidence to determine their true values.
Comparisons involving these attributes must report this limitation.

In [ ]:
# Create working copies, keeping the original tables unchanged.
customers_clean = customers.copy()
products_clean = products.copy()

# Flag missing information without guessing replacement values.
customers_clean["Job Title Missing"] = (
    customers_clean["Job Title"].isna()
)

products_clean["Color Missing"] = (
    products_clean["Color"].isna()
)

products_clean["Sizes Missing"] = (
    products_clean["Sizes"].isna()
)

print("Missing-value flags:")
print("Job Title:", customers_clean["Job Title Missing"].sum())
print("Color:", products_clean["Color Missing"].sum())
print("Sizes:", products_clean["Sizes Missing"].sum())

print("\nRows retained:")
print("Customers:", len(customers_clean))
print("Products:", len(products_clean))

#### Full Transaction Missing-Value Profiling

The complete transactions file is loaded once above. The following summary covers all rows before duplicate removal.

In [ ]:
display(missing_summary(transactions))

### Transaction Missing-Value Handling

Color is missing in 4,350,783 transaction rows (67.80%), and
Size is missing in 413,102 rows (6.44%).

We retain these rows because customer IDs, product IDs, and
transaction types remain available for the planned demographic
and product-category comparisons. Dropping rows with missing
attributes could substantially reduce coverage.

We preserve missing Color and Size values and flag them rather
than guessing replacements. Analyses involving these attributes
must acknowledge the incomplete information.

In [ ]:
# Add flags without copying the large transaction table.
# Original Color and Size values remain unchanged.
transactions["Color Missing"] = transactions["Color"].isna()
transactions["Size Missing"] = transactions["Size"].isna()

print(transactions[["Color Missing", "Size Missing"]].sum())

### 3.2 Other Quality Issues

#### Duplicate Checks

In [ ]:
print("Duplicate customer rows:", customers.duplicated().sum())
print("Duplicate Customer IDs:",
      customers["Customer ID"].duplicated().sum())

print("Duplicate product rows:", products.duplicated().sum())
print("Duplicate Product IDs:",
      products["Product ID"].duplicated().sum())

print("Duplicate transaction rows:",
      transactions.duplicated().sum())

print("Repeated invoice-line combinations:",
      transactions.duplicated(subset=["Invoice ID", "Line"]).sum())

### Duplicate Findings

Customers and products have no exact duplicate rows or repeated
primary IDs.

Transactions contain 798 exact duplicate rows and 22,312
additional occurrences of Invoice ID–Line combinations.
We inspect these records before removing anything because
invoice lines may recur across sales and returns.

In [ ]:
# examples of exact duplicates.
print("EXACT DUPLICATE EXAMPLES")
display(
    transactions.loc[transactions.duplicated()].head(5)
)

# Three repeated invoice-line combinations.
repeated_keys = (
    transactions.loc[
        transactions.duplicated(subset=["Invoice ID", "Line"]),
        ["Invoice ID", "Line"],
    ]
    .drop_duplicates()
    .head(3)
)

print("RECORDS SHARING THOSE INVOICE-LINE COMBINATIONS")
display(
    transactions.merge(
        repeated_keys,
        on=["Invoice ID", "Line"],
        how="inner",
    )[
        ["Invoice ID", "Line", "Customer ID", "Product ID",
         "Date", "Transaction Type", "Quantity", "Line Total"]
    ].sort_values(["Invoice ID", "Line", "Date"])
)

### Duplicate Handling

We remove 798 exact duplicate transaction rows, keeping the first
occurrence. We treat identical records across all supplied fields
as duplicate entries because no field distinguishes them as
separate events.

Repeated Invoice ID–Line combinations are not automatically
removed. Inspection shows that some refer to different products,
so deleting by these two fields could discard distinct records.
Remaining repeated combinations are flagged and retained because
their correct interpretation cannot be established from the data.

In [ ]:
rows_before = len(transactions)

transactions = transactions.drop_duplicates().reset_index(drop=True)

print("Exact duplicate rows removed:",
      rows_before - len(transactions))

print("Exact duplicates remaining:",
      transactions.duplicated().sum())

transactions["Invoice Line Review"] = transactions.duplicated(
    subset=["Invoice ID", "Line"],
    keep=False,
)

print("Rows retained:", len(transactions))
print("Rows flagged for invoice-line review:",
      transactions["Invoice Line Review"].sum())

#### Data Types and Date Validity

Inspect types and check date parsing before conversion. Unparseable values are counted for investigation.


In [ ]:
print("CUSTOMER DATA TYPES")
display(customers_clean.dtypes)

print("PRODUCT DATA TYPES")
display(products_clean.dtypes)

print("TRANSACTION DATA TYPES")
display(transactions.dtypes)

In [ ]:
# Parse dates into temporary variables.
birth_dates = pd.to_datetime(
    customers_clean["Date Of Birth"],
    format="%Y-%m-%d",
    errors="coerce",
)

transaction_dates = pd.to_datetime(
    transactions["Date"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
)

# Count values that were present but could not be parsed.
invalid_birth_dates = (
    customers_clean["Date Of Birth"].notna()
    & birth_dates.isna()
)

invalid_transaction_dates = (
    transactions["Date"].notna()
    & transaction_dates.isna()
)

print("Unparseable birth dates:", invalid_birth_dates.sum())
print("Unparseable transaction dates:",
      invalid_transaction_dates.sum())

print("\nTransaction date range:")
print(transaction_dates.min(), "to", transaction_dates.max())

In [ ]:
customers_clean["Date Of Birth"] = birth_dates
transactions["Date"] = transaction_dates

print("Birth-date type:", customers_clean["Date Of Birth"].dtype)
print("Transaction-date type:", transactions["Date"].dtype)

Both date columns were initially stored as text. All non-missing
values parsed successfully using their expected formats, so we
converted them to datetime types. Transactions cover 1 January
2023 to 18 March 2025. Successful parsing confirms valid date
formats, but does not establish whether every date is plausible.

In [ ]:
checks = pd.Series({
    "Birth dates after latest transaction date": (
        customers_clean["Date Of Birth"]
        > transactions["Date"].max()
    ).sum(),

    "Negative product production costs": (
        products_clean["Production Cost"] < 0
    ).sum(),

    "Negative transaction unit prices": (
        transactions["Unit Price"] < 0
    ).sum(),

    "Zero or negative transaction quantities": (
        transactions["Quantity"] <= 0
    ).sum(),

    "Discounts outside 0–1": (
        ~transactions["Discount"].between(0, 1)
    ).sum(),

    "Transaction types other than Sale or Return": (
        ~transactions["Transaction Type"].isin(["Sale", "Return"])
    ).sum(),
})

display(checks)

### Basic Validity Findings

No birth dates occur after the latest transaction date. Production
costs and unit prices are non-negative, quantities are positive,
discounts fall within 0–1, and transaction types are Sale or Return.

No changes were required for these checks. Negative return amounts
are retained because they represent refunds rather than invalid prices.

In [ ]:
text_checks = {
    "Customer Gender": customers_clean["Gender"],
    "Customer Country": customers_clean["Country"],
    "Product Category": products_clean["Category"],
    "Product Sub Category": products_clean["Sub Category"],
    "Product Color": products_clean["Color"],
    "Product Sizes": products_clean["Sizes"],
    "Transaction Size": transactions["Size"],
    "Transaction Color": transactions["Color"],
    "Transaction Type": transactions["Transaction Type"],
    "Payment Method": transactions["Payment Method"],
    "Currency": transactions["Currency"],
}

results = []

for name, series in text_checks.items():
    text = series.dropna().astype("string")
    labels = text.unique()

    results.append({
        "Field": name,
        "Rows with outer spaces": int(
            text.ne(text.str.strip()).sum()
        ),
        "Distinct labels": len(labels),
        "Labels after strip/lower": (
            pd.Series(labels)
            .str.strip()
            .str.lower()
            .nunique()
        ),
    })

display(pd.DataFrame(results).set_index("Field"))

### Category Formatting Findings

The inspected categorical fields contain no leading or trailing
spaces. Stripping spaces and converting labels to lowercase does
not reduce the number of distinct labels, so no category merges
or formatting changes are required for these checks.

#### Customer and Product Reference Checks

Check that transaction references exist in the customer and product tables.


In [ ]:
unmatched_customers = (
    ~transactions["Customer ID"].isin(customers_clean["Customer ID"])
)

unmatched_products = (
    ~transactions["Product ID"].isin(products_clean["Product ID"])
)

print("Transaction rows with unmatched Customer IDs:",
      unmatched_customers.sum())

print("Transaction rows with unmatched Product IDs:",
      unmatched_products.sum())

### Reference Check Findings

All retained transactions reference existing Customer IDs and
Product IDs. No rows require removal or flagging for unmatched
customer or product references.

In [ ]:
phones = customers_clean["Telephone"].dropna().astype("string")

phone_checks = pd.Series({
    "Contains spaces": phones.str.contains(" ", regex=False).sum(),
    "Contains hyphens": phones.str.contains("-", regex=False).sum(),
    "Contains parentheses": phones.str.contains(r"[()]").sum(),
    "Contains dots": phones.str.contains(".", regex=False).sum(),
    "Starts with +": phones.str.startswith("+").sum(),
    "Digits only": phones.str.fullmatch(r"[0-9]+").sum(),
})

display(phone_checks)

### Telephone Format Handling

Telephone values use multiple formats, including spaces, hyphens,
parentheses, dots, international prefixes, and digits-only strings.
The format counts overlap and do not establish number validity.

Telephone values were loaded as text to preserve leading zeros
and symbols. We retain them unchanged and exclude them from the
planned analysis because telephone numbers are not required for
demographic or product-category comparisons.

#### Missing Values and Other Quality Issues: Summary

| Issue | Finding | Handling |
|---|---|---|
| Missing customer occupations | 584,185 Job Title values missing (35.55%) | Retained missing values and added a flag |
| Missing product attributes | Color: 69.37%; Sizes: 11.54% | Retained products and added flags; no guessed replacements |
| Missing transaction attributes | Color: 67.80%; Size: 6.44%, before deduplication | Retained transactions and added flags |
| Exact duplicate transactions | 798 additional identical rows | Removed, assuming identical records represent duplicate entries |
| Repeated invoice-line combinations | 41,158 rows flagged after deduplication | Retained for review because some contain different products |
| Date types | Both date columns were text; all values parsed successfully | Converted to datetime |
| Basic validity checks | No violations of the six tested rules | No changes required |
| Category formatting | No outer spaces or case-only label variants in inspected fields | No changes required |
| Customer/product references | No unmatched transaction references | No changes required |
| Telephone formats | Multiple formats present | Preserved as text and excluded from analysis |

After removing exact duplicates, 6,416,029 transaction rows remain.
All 1,643,306 customers and 17,940 products are retained.

Missing product attributes and unresolved invoice-line conflicts
remain limitations. These checks support further analysis but do
not establish that the dataset is entirely error-free.

### 3.3 Outliers — Task C Pending

**Required contribution not yet present in the inspected Task C pull request.** Add Task C's actual detection code, results, and justified handling here. The uploaded Task C notebook currently repeats Task B and ends with table information; it contains no outlier detection or handling.

Continue using `customers_clean`, `products_clean` and the deduplicated `transactions` table above. Do not reload the CSVs or repeat Task B. Compare monetary values within currency, and distinguish Sale from Return when interpreting signed amounts. State the method and reason for keeping, capping or removing flagged values. Do not insert unverified results.

## 4. Exploratory Diagnostic Charts

### 4.1 Missing Product Attributes

This chart exposes missingness by product group and helps explain why rows with missing attributes are retained.

In [ ]:
import matplotlib.pyplot as plt

chart_data = product_missing_by_group[
    ["color_missing_pct", "sizes_missing_pct"]
].copy()

# Combine category and subcategory into readable labels.
chart_data.index = [
    f"{category} / {subcategory}"
    for category, subcategory in chart_data.index
]

chart_data.columns = ["Color missing", "Sizes missing"]

ax = chart_data.plot.barh(
    figsize=(12, 12),
    color=["#2878B5", "#E69F00"],
    width=0.8,
)

ax.set_title("Missing Product Attributes by Category and Subcategory")
ax.set_xlabel("Missing values within group (%)")
ax.set_ylabel("Product group")
ax.set_xlim(0, 105)
ax.invert_yaxis()
ax.legend(title="Attribute")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

Missing sizes are confined to Accessories, while missing colours vary substantially between subcategories. Deleting incomplete products would therefore disproportionately remove particular groups. Missing values are retained with flags rather than imputed without evidence.

### 4.2 Outlier Diagnostic Chart — Task C Pending

Insert at least one labeled chart that exposes the outlier issue investigated in Section 3.3, plus its interpretation. Together with the missingness chart, this satisfies the assignment's minimum of two diagnostic charts once completed.

## 5. PII and Bias Risks — Task D Pending

Insert Task D's assessment and any supporting code here. Distinguish generated names, contact details and demographic fields from real-person data: the publisher describes this dataset as synthetic. Explain handling of identity-like fields, uneven group representation, missingness patterns, and limits on applying simulated patterns to a real retailer. Do not claim the dataset establishes causal drivers of returns.

## 6. Conclusion — Task D Pending

State whether the data is fit for the scenario after all handling, supported by Tasks B and C. Include remaining missing attributes, unresolved invoice-line conflicts, outlier decisions, synthetic-data limitations and the limits of interpreting return records without a verified purchase-return linkage and denominator.

## Submission Check

- Complete all four member names and matric numbers.
- Replace pending sections with verified Tasks C and D.
- Confirm the dataset version and the final observations/decisions.
- Restart the kernel and run all cells in order; review actual results and at least two labeled diagnostic charts.
- Save the executed notebook, make it the repository's clearly identified submission notebook, and submit the repository link.
